# LabSD E1 — Cascade Degradation

Thin orchestrator. Heavy logic lives in the `labsd` package (Kaggle Dataset `ifty1011/labsd-src`).

## Phases
1. Env check + install
1.5. Extract nuScenes mini tarball to writable scratch
2. Build Boston/Singapore splits (sanity check on real data)
3. Train C1 (CenterPoint) on Boston
4. Train C2 (MTP) on Boston
5. Set up C3 (IDM planner)
6. Baseline 5-measurement run on Singapore
7. Retrain C1 on Singapore
8. Post-retrain 5-measurement run
9. Table I + cascade diagnostic + plot

## Required attached datasets
- `ifty1011/nuscenes-mini` — 4 GB tarball at `/kaggle/input/nuscenes-mini/v1.0-mini.tgz`
- `ifty1011/labsd-src` — our Python package, pip-installable

## Smoke-test mode
Set `SMOKE_TEST = True` (cell below) to run only Phases 1–2 and exit. Used for the first end-to-end validation before kicking off training.

In [ ]:
# ============================================================
# Run-mode flag
# ============================================================
SMOKE_TEST = True   # set False once env + extract + splits are validated

## Phase 1 — env check

In [ ]:
import os, sys, torch
print('python:', sys.version.split()[0])
print('torch :', torch.__version__)
print('cuda  :', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
print('---')
print('contents of /kaggle/input/:')
for entry in sorted(os.listdir('/kaggle/input')):
    sub = os.path.join('/kaggle/input', entry)
    inner = sorted(os.listdir(sub))[:8] if os.path.isdir(sub) else '(file)'
    print(f'  {entry}/ → {inner}')
print('---')
!df -h /kaggle/working

In [ ]:
# Phase 1 — make labsd importable, robust to dataset packing AND mount layout
import os, sys, tarfile, zipfile, shutil, glob, subprocess
from pathlib import Path

candidates = [
    '/kaggle/input/labsd-src',
    '/kaggle/input/datasets/ifty1011/labsd-src',
]
candidates += glob.glob('/kaggle/input/**/labsd-src', recursive=True)
SRC_INPUT = next((p for p in candidates if os.path.isdir(p)), None)
if SRC_INPUT is None:
    raise RuntimeError('labsd-src not found')
print('found SRC_INPUT =', SRC_INPUT)

LABSD_DIR = '/kaggle/working/labsd_pkg'
Path(LABSD_DIR).mkdir(parents=True, exist_ok=True)
if not Path(LABSD_DIR + '/labsd/__init__.py').exists():
    if Path(SRC_INPUT + '/labsd.tar').exists():
        with tarfile.open(SRC_INPUT + '/labsd.tar') as tf: tf.extractall(LABSD_DIR)
    elif Path(SRC_INPUT + '/labsd.zip').exists():
        with zipfile.ZipFile(SRC_INPUT + '/labsd.zip') as zf: zf.extractall(LABSD_DIR)
    elif Path(SRC_INPUT + '/labsd').is_dir():
        shutil.copytree(SRC_INPUT + '/labsd', LABSD_DIR + '/labsd', dirs_exist_ok=True)
if LABSD_DIR not in sys.path:
    sys.path.insert(0, LABSD_DIR)
import labsd
print('labsd loaded:', labsd.__file__)

# nuscenes-devkit: install with --no-deps to avoid downgrading Kaggle's
# numpy/scipy/sklearn. Then pull in the few extra deps it actually needs
# (mostly small pure-python packages that don't conflict).
print('\ninstalling nuscenes-devkit (no-deps) ...')
ret = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', 'nuscenes-devkit'],
    capture_output=True, text=True,
)
if ret.returncode != 0:
    print('STDOUT:', ret.stdout[-1500:]); print('STDERR:', ret.stderr[-1500:])
    raise RuntimeError('nuscenes-devkit install failed')
print(ret.stdout.strip().split('\n')[-1])

# Auxiliary deps not bundled in Kaggle's image. Pin pyquaternion which is tiny.
ret = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'pyquaternion', 'cachetools', 'descartes'],
    capture_output=True, text=True,
)
if ret.returncode != 0:
    print('STDERR:', ret.stderr[-1500:])
    raise RuntimeError('aux deps install failed')
print('aux deps OK')

# Smoke import — fail fast if anything is still missing
from nuscenes.nuscenes import NuScenes
import nuscenes
print('nuscenes-devkit version:', getattr(nuscenes, '__version__', 'unknown'))

## Phase 1.5 — extract nuScenes mini tarball

`/kaggle/input/` is read-only and contains the compressed `v1.0-mini.tgz`. We extract it once to `/kaggle/working/nuscenes/` (persistent across sessions when notebook persistence is enabled). Subsequent runs skip the extract.

In [ ]:
import os, tarfile, time, glob
from pathlib import Path

candidates = [
    '/kaggle/input/nuscenes-mini/v1.0-mini.tgz',
    '/kaggle/input/datasets/ifty1011/nuscenes-mini/v1.0-mini.tgz',
]
candidates += glob.glob('/kaggle/input/**/v1.0-mini.tgz', recursive=True)
TARBALL = next((p for p in candidates if os.path.exists(p)), None)
assert TARBALL, f'tarball missing. /kaggle/input/ tree:\n' + os.popen('find /kaggle/input -maxdepth 4').read()
print('found TARBALL =', TARBALL)

NUSC_ROOT = '/kaggle/working/nuscenes'
MARKER = Path(NUSC_ROOT) / '.extracted'
Path(NUSC_ROOT).mkdir(parents=True, exist_ok=True)

if MARKER.exists():
    print('already extracted, skipping.')
else:
    print(f'extracting {TARBALL} → {NUSC_ROOT} ...')
    t0 = time.time()
    with tarfile.open(TARBALL, 'r:gz') as tf:
        tf.extractall(NUSC_ROOT)
    MARKER.touch()
    print(f'done in {time.time()-t0:.1f}s')

!ls $NUSC_ROOT
print('---')
!ls $NUSC_ROOT/v1.0-mini/ 2>/dev/null | head -10

## Phase 2 — build Boston/Singapore splits

Sanity check on real metadata: confirms nuscenes-devkit can load the dataset and the location-based partition produces the expected ~6 Boston / ~4 Singapore.

In [ ]:
from labsd.splits import build_splits

splits = build_splits(
    nusc_root=NUSC_ROOT,
    out_path='/kaggle/working/splits/splits.json',
)
for k, v in splits.items():
    print(f'{k:18s}: {len(v):2d} scene(s)  {v}')

total = sum(len(v) for v in splits.values())
assert total == 10, f'expected 10 mini scenes, got {total}'
assert len(splits['boston_val']) >= 1
assert len(splits['singapore_val']) >= 1
print('\nsplits OK')

In [ ]:
# Smoke-test gate — stop here if SMOKE_TEST is on
if SMOKE_TEST:
    raise SystemExit('SMOKE_TEST: env + extract + splits all OK. Set SMOKE_TEST=False to continue.')

## Phase 3 — train C1 (CenterPoint) on Boston

In [ ]:
from labsd.train_c1 import train_c1
c1_boston = train_c1(
    config_path='/kaggle/input/labsd-src/configs/centerpoint_boston.py',
    work_dir='/kaggle/working/c1_boston',
)
print('C1 ckpt:', c1_boston)

## Phase 4 — train C2 (MTP) on Boston

In [ ]:
from labsd.train_c2 import train_c2
c2_boston = train_c2(
    split='boston_train',
    out_path='/kaggle/working/c2_boston.pth',
)
print('C2 ckpt:', c2_boston)

## Phase 6 — baseline 5-measurement run on Singapore

In [ ]:
from labsd.eval import run_all_measurements
baseline = run_all_measurements(
    c1_ckpt=c1_boston, c2_ckpt=c2_boston,
    split='singapore_val',
    out_path='/kaggle/working/baseline.json',
)
print(baseline)

## Phase 7 — retrain C1 on Singapore

In [ ]:
from labsd.train_c1 import train_c1
c1_singapore = train_c1(
    config_path='/kaggle/input/labsd-src/configs/centerpoint_singapore.py',
    work_dir='/kaggle/working/c1_singapore',
    resume_from=c1_boston,
)
print('C1 (Singapore-retrained):', c1_singapore)

## Phase 8 — post-retrain 5-measurement run

In [ ]:
after = run_all_measurements(
    c1_ckpt=c1_singapore, c2_ckpt=c2_boston,
    split='singapore_val',
    out_path='/kaggle/working/after_retrain.json',
)
print(after)

## Phase 9 — Table I + diagnostic + plot

In [ ]:
from labsd.table import build_table, cascade_diagnostic
df = build_table(
    baseline_json='/kaggle/working/baseline.json',
    after_json='/kaggle/working/after_retrain.json',
    out_csv='/kaggle/working/table_I.csv',
)
print(df.to_string(index=False))
print()
print('diagnostic:', cascade_diagnostic(df))

In [ ]:
import matplotlib.pyplot as plt

by_metric = {row['Metric']: row for _, row in df.iterrows()}
iso = by_metric['L2 w/ GT predictions']
pipe = by_metric['L2 w/ live C2']

labels = ['C3 (isolated)', 'C3 (pipeline)']
before = [iso['Before'], pipe['Before']]
after_vals = [iso['After'], pipe['After']]

fig, ax = plt.subplots(figsize=(7, 4.5))
x = range(len(labels))
ax.bar([i - 0.2 for i in x], before, 0.4, label='Boston-trained C1')
ax.bar([i + 0.2 for i in x], after_vals, 0.4, label='Singapore-retrained C1')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('L2 error (m)')
ax.set_title('Cascade effect on planning')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/cascade_result.png', dpi=150, bbox_inches='tight')
plt.show()